In [50]:
import numpy as np
import commpy as cp
# import matplotlib.pyplot as plt

In [51]:
M = 16
SNR = 35
D_noise = 10**(-SNR/10)

modem = cp.QAMModem(M)

input_bits = np.random.randint(0, 2, size=10000)

Модулированные символы:
$$
I= \{I_i \}_{i=0}^{N-1};
$$

In [52]:
input_signal = modem.modulate(input_bits) / np.sqrt(modem.Es)
N = len(input_signal)

Импульсная характеристика многолучевого канала:
$$
h=\{h_i \}_{i=0}^{L-1};
$$

In [53]:
L = 2
h = 10**(-np.linspace(0, 10, L) / 10)

Сигнал на выходе из канала:
$$\nu = h * I + n \quad \Longleftrightarrow \quad \nu_k = \sum\limits_{i=0}^{L-1}h_i \cdot I_{k-i} + n_k. \quad (k - i \geq 0)$$

In [54]:
output_signal = np.zeros_like(input_signal)
for k in range(N):
    for i in range(L):
        # Свёртка с ИХ канала
        if k-i >= 0:
            output_signal[k] += h[i] * input_signal[k-i]
    # Добавление шума
    # output_signal[k] += (np.random.randn() + 1j * np.random.randn()) * np.sqrt(D_noise / 2)

Запускаем сигнал на фильтр прямой связи (**Feed-Forward Filter**):
$$
y = w^{ff} * \nu \quad \Longleftrightarrow \quad y_k = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot \nu_{k-i}, \quad (k - i \geq 0)
$$
где $w^{ff}$ - веса фильтра прямой связи.
Для определения фильтра с обратной связью по решению, рассмотрим поподробнее сигнал на выходе из фильтра прямой связи:
$$
\begin{aligned}
    y_k = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot \nu_{k-i} = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot (\sum\limits_{l=0}^{L-1}h_l \cdot I_{k-i-l} + n_{k-i})&= w^{ff}_0h_0I_k + w^{ff}_0h_1I_{k-1} + w^{ff}_0h_2I_{k-2} + \ldots + \\
    &+w^{ff}_1h_0I_{k-1} + w^{ff}_1h_1I_{k-2} + w^{ff}_1h_2I_{k-3} + \ldots + \\
    &+w^{ff}_2h_0I_{k-2} + w^{ff}_2h_1I_{k-3} + w^{ff}_2h_2I_{k-4} + \ldots + \\
    &+ \ldots + \\
    &+\sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} = \\
    &= w^{ff}_0h_0I_k + (w^{ff}_0h_1 + w^{ff}_1h_0)\cdot I_{k-1} + \\
    &+ (w^{ff}_0h_2 + w^{ff}_1h_1 + w^{ff}_2h_0) \cdot I_{k-2} + \\
    &+ (w^{ff}_0h_3 + w^{ff}_1h_2 + w^{ff}_2h_1 + w^{ff}_3h_0) \cdot I_{k-3} + \\
    &+\ldots+ \\
    &+\sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} = \\
    &=w^{ff}_0h_0I_k + \sum_{i=1}^{K_2-1}I_{k-i}\sum_{l=0}^i w^{ff}_lh_{i-l} + \sum_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i},
\end{aligned}
$$
где $K_2 = 2(K_1 - 1) + 1$.

Получаем сумму основного сигнала и прекурсоров. Для того чтобы избавиться от них, пропустим сигнал через фильтр с обратной связью по решению (**Feed-Back Filter**):
$$
\hat{I} = y - w^{bf} * \hat{I}' \quad \Longleftrightarrow \quad \hat{I}_k = y_k - \sum_{i=1}^{K_2-1}w^{bf}_i \cdot \hat{I}_{k-i}'.
$$
Тогда эквализированный сигнал будет выглядеть следующим образом:
$$
\hat{I}_k = w^{ff}_0h_0I_k + \sum_{i=1}^{K_2-1}\left(I_{k-i}\sum_{l=0}^i w^{ff}_lh_{i-l} - \hat{I}_{k-i} \cdot w^{bf}_i \right) + \sum_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} \quad \Rightarrow \quad w^{bf}_i = \sum_{l=0}^i w^{ff}_lh_{i-l}.
$$
Теперь определим веса фильтра прямой связи по критерию пикового искажения (**Zero-Forcing**):
$$
w^{ff} = \{ w^{ff}_i \}_{i=0}^{K_1-1} = \{ 1 / h_i \}_{i=0}^{L-1}.
$$
Формулы эквализированного сигнала и весов фильтра обратной связи по решению будут выглядеть следующий образом:
$$
\hat{I}_k = I_k + \sum_{i=1}^{2L-2}\left(I_{k-i}\sum_{l=0}^i h_{i-l}/h_l - \hat{I}_{k-i} \cdot w^{bf}_i \right) + \sum_{i=0}^{L-1}n_{k-i}/h_i; \quad w^{bf} = \{ w^{bf}_i \}_{i=1}^{2L-2} = \{ \sum_{l=0}^i h_{i-l}/h_l \}_{i=1}^{2L-2}. \quad (0 \leq i-l \leq L-1)
$$

In [55]:
# Количество ячеек памяти Feed-Forward Filter
K1 = len(h)
# Веса Feed-Forward Filter по критерию Zero-Forcing
w_ff = (h**(-1))

# Количество ячеек памяти Feed-Back Filter
K2 = 2 * (K1 - 1) + 1
# Веса Feed-Back Filter
w_bf = np.zeros(K2)
for i in range(1, K2):
    for l in range(K1):
        if 0 <= i-l < len(h):
            w_bf[i] += w_ff[l] * h[i-l]
        else:
            continue

# Сигнал после прямого фильтра
y = np.zeros_like(output_signal)
for k in range(N):
    for i in range(K1):
        if k-i >= 0:
            y[k] += w_ff[i] * output_signal[k-i]
        else:
            break
I_hat = np.zeros_like(y)
for k in range(len(output_signal)):
    for i in range(1, K2):
        if k - i >= 0:
            y[k] -= w_bf[i] * I_hat[k-i]
        else:
            break
    I_hat[k] = modem.modulate(modem.demodulate(y[k]*np.sqrt(modem.Es), demod_type='hard'))[0] / np.sqrt(modem.Es)

output_bits = modem.demodulate(I_hat*np.sqrt(modem.Es), demod_type='hard')

ber = np.mean((input_bits + output_bits) % 2)

print(ber)

0.0


Векторизуем задачу для упрощения вычислений. Построим вектора, характеризующие канал, а именно импульсную характеристику и АБГШ:
$$
\vec{h} = (h_{i})_{0 \leq i \leq L - 1}, \quad \vec{n} = (n_{i})_{0 \leq i \leq N - 1}
$$
Построим вектор прекурсоров с основным символом:
$$
\vec{I}_k = (I_{i})_{0 \leq i \leq L - 1}, \quad I_i = \left\{
\begin{aligned}
& I_{k-i}, \quad k \geq L-1 \\
& 0, \quad \text{иначе}
\end{aligned}
\right.
$$
Чтобы перейти от свёртки с импульсной характеристикой канала к скалярному произведению, сделали его инвертированным. Теперь мы можем переписать вектор сигнала на выходе из канала:
$$
\vec{\nu} = (\vec{I}_k \cdot \vec{h} + n_k)_{0 \leq k \leq N - 1}
$$
Получим сигнал на выходе из фильтра прямой связи. Для этого возьмём срез сигнала на выходе из канала длиной $K_1$ и инвертируем его:
$$
\vec{\nu}_k = (\nu_{i})_{0 \leq i \leq K_1 - 1}, \quad \nu_i = \left\{
\begin{aligned}
& \nu_{k-i}, \quad k \geq K_1-1 \\
& 0, \quad \text{иначе}
\end{aligned}
\right.
$$
Получаем:
$$
\vec{f} = (\vec{\nu}_k \cdot \vec{w}^{ff})_{0 \leq k \leq N - 1}
$$
где $\vec{w}^{ff}$ - вектор весов фильтра прямой связи:
$$
\vec{w}^{ff} = (w_{i}^{ff})_{0 \leq i \leq K_1 - 1}
$$
Работа фильтра обратной связи по решению характеризуется свёрткой весов фильтра и ранее продетектированных символов:
$$
\vec{b} = (\vec{\hat{I}}_k \cdot \vec{w}^{bf})_{0 \leq k \leq N - 1}
$$
где вектор ранее продетектированных символов $\vec{\hat{I}}_k$:
$$
\vec{\hat{I}}_k = (\hat{I}_{i})_{0 \leq i \leq K_2 - 1}, \quad \hat{I}_i = \left\{
\begin{aligned}
& \hat{I}_{k-i + 1}, \quad k \geq K_2-1 \\
& 0, \quad \text{иначе}
\end{aligned}
\right.
$$
и весов фильтра обратной связи по решению $\vec{w}^{bf}$:
$$
\vec{w}^{bf} = (w_{i}^{bf})_{0 \leq i \leq K_2 - 1}
$$
Получаем эквализированный сигнал:
$$
\vec{y} = \vec{f} - \vec{b}
$$
Построим матрицу $\underline{\underline{h}}$:
$$
\underline{\underline{h}} = (h_{ij})_{\begin{aligned} & 0 \leq i \leq \min(k, \ L-1) \\ & 0 \leq j \leq \min(k, \ L-1) \end{aligned}}
$$
Для того чтобы она стала свёрточной матрицей, нужно наложить следующее условие:
$$
h_{ij} = \left\{
\begin{aligned}
& h_j, \quad j \leq i \\
& 0, \quad \text{иначе}
\end{aligned}
\right.
$$
Тогда:
$$
\underline{\underline{h}} = \begin{pmatrix}
h_0 & 0 & \cdots & 0 \\
h_0 & h_1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
h_0 & h_1 & \cdots & h_{L-1}
\end{pmatrix}
$$